In [0]:
from pyspark.sql import functions as F

def ip_to_int_expr(col_name):
    parts = F.split(F.col(col_name), "\\.")
    return (
        parts.getItem(0).cast("long") * 16777216 +
        parts.getItem(1).cast("long") * 65536 +
        parts.getItem(2).cast("long") * 256 +
        parts.getItem(3).cast("long")
    )

In [0]:
blocks_df = spark.table("fraud_detection.bronze.ip_geolocation_blocks_raw")

blocks_parsed = (
    blocks_df
    .withColumn("network_ip", F.split(F.col("network"), "/").getItem(0))
    .withColumn("prefix_length", F.split(F.col("network"), "/").getItem(1).cast("int"))
    .withColumn("network_int", ip_to_int_expr("network_ip"))
    .withColumn("block_size", F.pow(F.lit(2), (32 - F.col("prefix_length"))).cast("long"))
    .withColumn("start_ip_int", F.col("network_int"))
    .withColumn("end_ip_int", F.col("network_int") + F.col("block_size") - F.lit(1))
    .select("start_ip_int", "end_ip_int", "geoname_id")
)

display(blocks_parsed.limit(10))

In [0]:
transactions_df = spark.table("fraud_detection.bronze.transactions_raw")

transactions_with_ip_int = transactions_df.withColumn(
    "ip_int", ip_to_int_expr("ip_address")
)

display(transactions_with_ip_int.select("ip_address", "ip_int").limit(5))

In [0]:
from pyspark.sql import functions as F

def ip_to_int_expr(col_name):
    parts = F.split(F.col(col_name), "\\.")
    return (
        parts.getItem(0).cast("long") * 16777216 +
        parts.getItem(1).cast("long") * 65536 +
        parts.getItem(2).cast("long") * 256 +
        parts.getItem(3).cast("long")
    )

blocks_df = spark.table("fraud_detection.bronze.ip_geolocation_blocks_raw")

blocks_parsed = (
    blocks_df
    .withColumn("network_ip", F.split(F.col("network"), "/").getItem(0))
    .withColumn("prefix_length", F.split(F.col("network"), "/").getItem(1).cast("int"))
    .withColumn("network_int", ip_to_int_expr("network_ip"))
    .withColumn("block_size", F.pow(F.lit(2), (32 - F.col("prefix_length"))).cast("long"))
    .withColumn("start_ip_int", F.col("network_int"))
    .withColumn("end_ip_int", F.col("network_int") + F.col("block_size") - F.lit(1))
    .select("start_ip_int", "end_ip_int", "geoname_id")
)

# bucket size = 65536 (a /16 worth of addresses)
BUCKET_SIZE = 65536

blocks_bucketed = (
    blocks_parsed
    .withColumn("start_bucket", (F.col("start_ip_int") / BUCKET_SIZE).cast("long"))
    .withColumn("end_bucket", (F.col("end_ip_int") / BUCKET_SIZE).cast("long"))
    # explode: one row per bucket this block touches (usually just 1 row, rarely more)
    .withColumn("bucket", F.explode(F.sequence(F.col("start_bucket"), F.col("end_bucket"))))
    .select("bucket", "start_ip_int", "end_ip_int", "geoname_id")
)

print(f"Blocks after bucket explosion: {blocks_bucketed.count()}")  # should be close to 3,755,595, not wildly larger

In [0]:
transactions_df = spark.table("fraud_detection.bronze.transactions_raw")

transactions_with_bucket = (
    transactions_df
    .withColumn("ip_int", ip_to_int_expr("ip_address"))
    .withColumn("bucket", (F.col("ip_int") / BUCKET_SIZE).cast("long"))
)

joined_df = transactions_with_bucket.join(
    F.broadcast(blocks_bucketed),
    (transactions_with_bucket.bucket == blocks_bucketed.bucket) &
    (transactions_with_bucket.ip_int >= blocks_bucketed.start_ip_int) &
    (transactions_with_bucket.ip_int <= blocks_bucketed.end_ip_int),
    "left"
)

print(f"Row count after join: {joined_df.count()}")

In [0]:
locations_df = spark.table("fraud_detection.bronze.ip_geolocation_locations_raw")

geo_enriched_df = joined_df.join(
    locations_df.select("geoname_id", "country_iso_code", "country_name", "subdivision_1_name", "city_name", "time_zone"),
    on="geoname_id",
    how="left"
)

display(geo_enriched_df.select(
    "transaction_id", "ip_address", "customer_location", 
    "country_name","country_iso_code", "subdivision_1_name", "city_name"
).limit(10))

print(f"Row count after locations join: {geo_enriched_df.count()}")

In [0]:
print(f"Row count after locations join: {geo_enriched_df.count()}")

In [0]:
final_geo_df = geo_enriched_df.select(
    "transaction_id", "customer_id", "transaction_amount", "transaction_date",
    "payment_method", "product_category", "quantity", "customer_age",
    "customer_location", "device_used", "ip_address", "shipping_address",
    "billing_address", "is_fraudulent", "account_age_days", "transaction_hour",
    "country_name", "country_iso_code", "subdivision_1_name", "city_name", "time_zone"
)

final_geo_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("fraud_detection.silver.transactions_geo_enriched")

display(spark.sql("DESCRIBE fraud_detection.silver.transactions_geo_enriched"))